# Week 9: Agent Frameworks — LangGraph & Strands Agents

**Lab companion to [week_09.md](../agenda/week_09.md)**

In this lab you will:
1. Build a LangGraph workflow (prompt chaining + routing)
2. Build a Strands Agents **Agents-as-Tools** pipeline
3. Build a Strands **Swarm** with autonomous handoffs
4. Build a Strands **Graph** with parallel branches and conditional edges
5. Use the Strands **Workflow** tool with task dependencies
6. *(Bonus)* Expose and consume an agent via **A2A**

In [ ]:
# Install frameworks
!pip install langgraph langchain-openai langchain-core strands-agents strands-agents-tools -q
# For A2A bonus exercise:
# !pip install 'strands-agents[a2a]' 'strands-agents-tools[a2a_client]' -q

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"
print("Ready!")

---
## Part 1: LangGraph — Prompt Chaining Workflow

Build a 3-step pipeline: **outline → draft → polish**.

Each node is a Python function that receives `state` (a `TypedDict`) and returns updated keys.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict

llm = ChatOpenAI(model="gpt-5-mini")

class BlogState(TypedDict):
    topic: str
    outline: str
    draft: str
    final: str

def create_outline(state: BlogState) -> dict:
    r = llm.invoke(f"Create a 3-point outline for a blog post about: {state['topic']}")
    return {"outline": r.content}

def write_draft(state: BlogState) -> dict:
    r = llm.invoke(f"Write a 150-word blog post from this outline:\n{state['outline']}")
    return {"draft": r.content}

def polish(state: BlogState) -> dict:
    r = llm.invoke(f"Improve this draft — fix grammar and flow:\n{state['draft']}")
    return {"final": r.content}

graph = StateGraph(BlogState)
graph.add_node("outline", create_outline)
graph.add_node("draft",   write_draft)
graph.add_node("polish",  polish)
graph.add_edge(START, "outline")
graph.add_edge("outline", "draft")
graph.add_edge("draft",   "polish")
graph.add_edge("polish",  END)
app = graph.compile()

result = app.invoke({"topic": "why AI agents are changing software development"})
print(result["final"])

### Exercise 1.1 — LangGraph Routing

Add a **routing node** before the outline step. If the topic contains a question mark, route to a `qa_node` that answers directly. Otherwise, route to `create_outline`.

Hint: use `graph.add_conditional_edges("router", routing_fn, {"outline": "outline", "qa": "qa_node"})`

In [ ]:
# TODO: Build a routed LangGraph
# 1. Add a 'router' node that classifies the input
# 2. Add a 'qa_node' for direct question answering
# 3. Use add_conditional_edges to route based on classification

# Test with:
# app.invoke({"topic": "What is a transformer model?"})
# app.invoke({"topic": "Benefits of RAG in production"})

<details>
<summary>Hint</summary>

```python
def router(state):
    if "?" in state["topic"]:
        return {"route": "qa"}
    return {"route": "outline"}

def routing_fn(state):
    return state["route"]

graph.add_node("router", router)
graph.add_conditional_edges("router", routing_fn, {"qa": "qa_node", "outline": "outline"})
```

Update `BlogState` to include `route: str`.
</details>

<details>
<summary>Solution</summary>

```python
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict

llm = ChatOpenAI(model="gpt-5-mini")

class BlogState(TypedDict):
    topic: str
    route: str
    outline: str
    draft: str
    final: str

def router(state):
    return {"route": "qa" if "?" in state["topic"] else "outline"}

def qa_node(state):
    r = llm.invoke(f"Answer this question concisely: {state['topic']}")
    return {"final": r.content}

def create_outline(state): ...
def write_draft(state): ...
def polish(state): ...

graph = StateGraph(BlogState)
graph.add_node("router",  router)
graph.add_node("qa_node", qa_node)
graph.add_node("outline", create_outline)
graph.add_node("draft",   write_draft)
graph.add_node("polish",  polish)

graph.add_edge(START, "router")
graph.add_conditional_edges("router", lambda s: s["route"],
                            {"qa": "qa_node", "outline": "outline"})
graph.add_edge("qa_node", END)
graph.add_edge("outline", "draft")
graph.add_edge("draft",   "polish")
graph.add_edge("polish",  END)
app = graph.compile()
```
</details>

---
## Part 2: Strands Agents — Multi-Agent Patterns

Strands Agents provides 5 multi-agent patterns. We'll practice 4 of them.

### Exercise 2.1 — Agents as Tools

Wrap two specialist agents as `@tool` functions and build an orchestrator that calls them in sequence.

In [ ]:
from strands import Agent, tool

# TODO: Create two @tool specialist agents:
# 1. summarizer(text: str) -> str
#    - Creates an Agent that summarizes text in 2 sentences
#
# 2. translator(text: str) -> str
#    - Creates an Agent that translates text to Spanish
#
# Then build an orchestrator Agent with both tools.
# Test: orchestrator("Summarize then translate to Spanish: 'LangGraph is a stateful agent framework built on LangChain.'")

<details>
<summary>Hint</summary>

```python
@tool
def summarizer(text: str) -> str:
    """Summarize text in 2 sentences."""
    agent = Agent(system_prompt="Summarize in exactly 2 sentences.")
    return str(agent(text))
```

The orchestrator's `system_prompt` should tell it to call `summarizer` first, then `translator`.
</details>

<details>
<summary>Solution</summary>

```python
from strands import Agent, tool

@tool
def summarizer(text: str) -> str:
    """Summarize text in 2 sentences."""
    agent = Agent(system_prompt="Summarize the given text in exactly 2 sentences. Be concise.")
    return str(agent(text))

@tool
def translator(text: str) -> str:
    """Translate text to Spanish."""
    agent = Agent(system_prompt="Translate the given text to Spanish. Return only the translation.")
    return str(agent(text))

orchestrator = Agent(
    system_prompt="""You are a pipeline orchestrator.
For every request:
1. Call 'summarizer' on the input text
2. Call 'translator' on the summary
Return the final Spanish summary.""",
    tools=[summarizer, translator]
)

result = orchestrator(
    "Summarize then translate to Spanish: "
    "'LangGraph is a stateful agent framework built on LangChain. "
    "It uses graphs with nodes and edges to orchestrate complex AI workflows.'"
)
print(result)
```
</details>

### Exercise 2.2 — Swarm

Build a `Swarm` of three agents that collaborate autonomously to design a REST API. The swarm decides its own handoff order.

In [ ]:
from strands import Agent
from strands.multiagent import Swarm

# TODO: Create a Swarm with three specialized agents:
# - architect: designs the API endpoints and data models
# - coder:     writes Python FastAPI code for the design
# - reviewer:  reviews the code and suggests improvements
#
# Configure: entry_point=architect, max_handoffs=10, max_iterations=10
#
# Run: swarm("Design and implement a simple REST API for a todo app with CRUD operations.")
# Print the agent sequence from result.node_history

<details>
<summary>Hint</summary>

```python
architect = Agent(
    name="architect",
    system_prompt="You design REST API structures. List endpoints and data models."
)
```

Each agent can call `handoff_to_agent(agent_name=..., message=...)` when it decides to pass to a specialist. The swarm provides this tool automatically.
</details>

<details>
<summary>Solution</summary>

```python
from strands import Agent
from strands.multiagent import Swarm

architect = Agent(
    name="architect",
    system_prompt="""You are a REST API architect.
Design clear API endpoints and data models.
When done, hand off to 'coder' for implementation."""
)

coder = Agent(
    name="coder",
    system_prompt="""You are a Python FastAPI specialist.
Implement the API design you receive.
When done, hand off to 'reviewer' for code review."""
)

reviewer = Agent(
    name="reviewer",
    system_prompt="""You are a code reviewer.
Review the FastAPI code for correctness, security, and best practices.
Provide specific improvement suggestions."""
)

swarm = Swarm(
    [architect, coder, reviewer],
    entry_point=architect,
    max_handoffs=10,
    max_iterations=10,
    execution_timeout=180.0,
)

result = swarm("Design and implement a simple REST API for a todo app with CRUD operations.")

print(f"Status: {result.status}")
print(f"Agent sequence: {[n.node_id for n in result.node_history]}")
```
</details>

### Exercise 2.3 — Graph (Parallel + Conditional)

Build a `GraphBuilder` pipeline:
- A `coordinator` fans out to two parallel workers
- Both workers feed into an `aggregator`
- A `reviewer` checks the result
- Only if the review says "approved" does it proceed to `publisher`

In [ ]:
from strands import Agent
from strands.multiagent import GraphBuilder

# TODO: Build a Graph with this shape:
#
#   coordinator
#    /        \
# worker_a  worker_b     (parallel)
#    \        /
#   aggregator
#       |
#    reviewer
#    /       \
# publisher  (back to aggregator if not approved)
#
# Steps:
# 1. Create 5 Agents: coordinator, worker_a, worker_b, aggregator, reviewer, publisher
# 2. Use GraphBuilder to wire the graph
# 3. Add a conditional edge: reviewer -> publisher only if "approved" in result
# 4. Set max_node_executions=6, entry_point="coordinator"

# Test: graph("Research the pros and cons of microservices vs monoliths")

<details>
<summary>Hint</summary>

```python
builder = GraphBuilder()
builder.add_node(coordinator, "coordinator")
builder.add_node(worker_a,    "worker_a")
# ...
builder.add_edge("coordinator", "worker_a")
builder.add_edge("coordinator", "worker_b")   # parallel!
builder.add_edge("worker_a",    "aggregator")
builder.add_edge("worker_b",    "aggregator")

# Conditional edge:
def is_approved(state):
    result = state.results.get("reviewer")
    return result and "approved" in str(result.result).lower()

builder.add_edge("reviewer", "publisher", condition=is_approved)
```
</details>

<details>
<summary>Solution</summary>

```python
from strands import Agent
from strands.multiagent import GraphBuilder

coordinator = Agent(name="coordinator",
    system_prompt="Decompose the task into two independent sub-tasks for worker_a and worker_b.",
    callback_handler=None)
worker_a = Agent(name="worker_a",
    system_prompt="Research the PROS side of the topic you receive.",
    callback_handler=None)
worker_b = Agent(name="worker_b",
    system_prompt="Research the CONS side of the topic you receive.",
    callback_handler=None)
aggregator = Agent(name="aggregator",
    system_prompt="Combine the pros and cons into a balanced analysis.",
    callback_handler=None)
reviewer = Agent(name="reviewer",
    system_prompt="Review the analysis. Reply with 'approved' if it is balanced and complete. Otherwise say 'needs revision'.",
    callback_handler=None)
publisher = Agent(name="publisher",
    system_prompt="Format the analysis as a polished 300-word article.")

def is_approved(state):
    r = state.results.get("reviewer")
    return bool(r and "approved" in str(r.result).lower())

builder = GraphBuilder()
builder.add_node(coordinator, "coordinator")
builder.add_node(worker_a,    "worker_a")
builder.add_node(worker_b,    "worker_b")
builder.add_node(aggregator,  "aggregator")
builder.add_node(reviewer,    "reviewer")
builder.add_node(publisher,   "publisher")

builder.add_edge("coordinator", "worker_a")
builder.add_edge("coordinator", "worker_b")
builder.add_edge("worker_a",    "aggregator")
builder.add_edge("worker_b",    "aggregator")
builder.add_edge("aggregator",  "reviewer")
builder.add_edge("reviewer",    "publisher",   condition=is_approved)

builder.set_entry_point("coordinator")
builder.set_max_node_executions(8)
builder.set_execution_timeout(180)
graph = builder.build()

result = graph("Analyze the pros and cons of microservices vs monoliths")
print(result.results["publisher"].result)
print(f"Execution order: {[n.node_id for n in result.execution_order]}")
```
</details>

### Exercise 2.4 — Workflow Tool

Use the built-in `workflow` tool to define a 3-step data analysis pipeline with explicit task dependencies.

In [ ]:
from strands import Agent
from strands_tools import workflow

agent = Agent(tools=[workflow])

# TODO: Use agent.tool.workflow() to:
#
# 1. action="create", workflow_id="content_pipeline"
#    Tasks:
#    - "research":  research AI trends (no dependencies)
#    - "analysis":  analyze the research (depends on research)
#    - "article":   write a 200-word article (depends on analysis)
#
# 2. action="start", workflow_id="content_pipeline"
#
# 3. action="status", workflow_id="content_pipeline" — print status
#
# Bonus: pause and resume the workflow

<details>
<summary>Hint</summary>

Task schema:
```python
{
    "task_id": "research",
    "description": "Research the top AI trends in 2025",
    "system_prompt": "You research AI topics and summarize findings.",
    "priority": 5
}
```

For dependent tasks, add: `"dependencies": ["research"]`
</details>

<details>
<summary>Solution</summary>

```python
from strands import Agent
from strands_tools import workflow

agent = Agent(tools=[workflow])

# Create workflow
agent.tool.workflow(
    action="create",
    workflow_id="content_pipeline",
    tasks=[
        {
            "task_id": "research",
            "description": "Research the top AI trends in 2025",
            "system_prompt": "You research AI topics and provide detailed summaries.",
            "priority": 5
        },
        {
            "task_id": "analysis",
            "description": "Analyze the research findings and identify the 3 most impactful trends",
            "dependencies": ["research"],
            "system_prompt": "You analyze research data and extract key insights.",
            "priority": 3
        },
        {
            "task_id": "article",
            "description": "Write a 200-word article about the key AI trends for a developer audience",
            "dependencies": ["analysis"],
            "system_prompt": "You write clear technical articles for developers.",
            "priority": 2
        }
    ]
)

# Start
agent.tool.workflow(action="start", workflow_id="content_pipeline")

# Check status
status = agent.tool.workflow(action="status", workflow_id="content_pipeline")
print(status)
```
</details>

---
### Bonus: A2A (Agent-to-Agent Protocol)

A2A lets you expose a Strands agent as a network service (HTTP) and consume it remotely. The client side looks identical to calling a local agent.

**Run the server in a separate terminal:**
```bash
pip install 'strands-agents[a2a]' strands-agents-tools -q
python - <<'EOF'
from strands import Agent
from strands_tools import calculator
from strands.multiagent.a2a import A2AServer

agent = Agent(
    name="Calculator Agent",
    description="Performs arithmetic operations.",
    tools=[calculator],
    callback_handler=None
)
A2AServer(agent=agent).serve()   # http://localhost:9000
EOF
```

**Then run the client below:**

In [ ]:
# Requires the A2A server running on localhost:9000
# Uncomment to run:

# from strands.agent.a2a_agent import A2AAgent
#
# remote_calc = A2AAgent(endpoint="http://localhost:9000")
#
# # Get agent metadata
# import asyncio
# card = asyncio.run(remote_calc.get_agent_card())
# print(f"Remote agent: {card.name}")
# print(f"Skills: {card.skills}")
#
# # Call it
# result = remote_calc("What is 1337 * 42?")
# print(result.message)
#
# # Use as a tool in an orchestrator
# from strands import Agent, tool
#
# @tool
# def calculate(expression: str) -> str:
#     """Perform a calculation via remote calculator agent."""
#     result = remote_calc(expression)
#     return str(result.message["content"][0]["text"])
#
# orchestrator = Agent(
#     system_prompt="You are helpful. Use 'calculate' for all math.",
#     tools=[calculate]
# )
# orchestrator("What is the area of a circle with radius 7? Use pi=3.14159")

---
## Summary

You practiced:

| Pattern | When to use |
|---------|-------------|
| **LangGraph** | Deterministic workflows, checkpointing, human-in-the-loop |
| **Strands — Agents as Tools** | Structured pipeline, task order known upfront |
| **Strands — Swarm** | Open-ended tasks, autonomous collaboration |
| **Strands — Graph** | Parallel branches, conditional routing, feedback loops |
| **Strands — Workflow** | Long-running jobs, explicit dependencies, pause/resume |
| **Strands — A2A** | Distributed agents across services |

**Next:** [Week 10 — LangGraph in Practice + LangSmith & Langfuse](week_10_frameworks_observability.ipynb)